# dlt Homework Notebook (Zoomcamp 2026)

Based on `dlt_homework.md`.

This notebook is hands-on and learning-first:
- build a custom API source (no scaffold)
- run a dlt pipeline into DuckDB
- answer the 3 homework questions with SQL

You can run this from your `dataTalks` conda environment.

## Run order

1. Open terminal: `conda activate dataTalks`
2. Start Jupyter and open this notebook.
3. Run cells top to bottom.

If a command fails, read the printed error and rerun after fixing it.

In [1]:
from pathlib import Path
import subprocess
import sys
import textwrap

# Find repo root robustly (walk upward until we find .git)
cur = Path.cwd().resolve()
REPO_ROOT = None
for p in [cur] + list(cur.parents):
    if (p / ".git").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (.git). Start notebook inside the zoomcamp repo.")

WORK_DIR = REPO_ROOT / "cohorts" / "2026" / "workshops" / "dlt" / "taxi_pipeline_project"
WORK_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_FILE = WORK_DIR / "taxi_pipeline.py"
DUCKDB_FILE = WORK_DIR / "taxi_pipeline.duckdb"

print("Repo root:", REPO_ROOT)
print("Project dir:", WORK_DIR)
print("DuckDB file:", DUCKDB_FILE)

Repo root: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp
Project dir: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/cohorts/2026/workshops/dlt/taxi_pipeline_project
DuckDB file: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/cohorts/2026/workshops/dlt/taxi_pipeline_project/taxi_pipeline.duckdb


In [2]:
# Optional: install deps in current kernel if needed
# Uncomment if you see ModuleNotFoundError for dlt/duckdb/requests
# %pip install "dlt[duckdb]" duckdb requests pandas

import importlib
for pkg in ["dlt", "duckdb", "requests", "pandas"]:
    try:
        importlib.import_module(pkg)
        print(f"OK: {pkg}")
    except Exception as e:
        print(f"MISSING: {pkg} -> {e}")

OK: dlt
OK: duckdb
OK: requests
OK: pandas


## Step 1: Create the pipeline code

This writes a full `taxi_pipeline.py` file with:
- custom paginated API logic (`page=1,2,3,...`)
- stop condition when API returns empty page
- dlt pipeline that loads to DuckDB

In [3]:
pipeline_code = textwrap.dedent(f'''
import requests
import dlt

BASE_URL = "https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api"
DUCKDB_PATH = r"{DUCKDB_FILE}"


def fetch_pages(page_size: int = 1000):
    page = 1
    while True:
        response = requests.get(
            BASE_URL,
            params={{"page": page, "page_size": page_size}},
            timeout=30,
        )
        response.raise_for_status()
        rows = response.json()
        if not rows:
            break
        yield from rows
        page += 1


@dlt.resource(name="ny_taxi")
def ny_taxi_resource():
    yield from fetch_pages(page_size=1000)


def run():
    pipeline = dlt.pipeline(
        pipeline_name="taxi_pipeline",
        destination=dlt.destinations.duckdb(DUCKDB_PATH),
        dataset_name="taxi_data",
    )
    load_info = pipeline.run(ny_taxi_resource(), write_disposition="replace")
    print(load_info)


if __name__ == "__main__":
    run()
''')

PIPELINE_FILE.write_text(pipeline_code)
print(f"Wrote: {PIPELINE_FILE}")
print(f"Pipeline will write to: {DUCKDB_FILE}")

Wrote: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/cohorts/2026/workshops/dlt/taxi_pipeline_project/taxi_pipeline.py
Pipeline will write to: /Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/cohorts/2026/workshops/dlt/taxi_pipeline_project/taxi_pipeline.duckdb


## Step 2: Run the pipeline

This may take a bit. It will create a local DuckDB file in the project folder and load the full taxi dataset.

If you get a DuckDB lock error, close any other Python/Jupyter process using the same DB file and rerun this cell.

In [4]:
cmd = [sys.executable, str(PIPELINE_FILE)]
result = subprocess.run(cmd, cwd=str(WORK_DIR), capture_output=True, text=True)
print("exit code:", result.returncode)
print("\nSTDOUT:\n", result.stdout[:4000])
print("\nSTDERR:\n", result.stderr[:4000])

if result.returncode != 0:
    err = result.stderr.lower()
    if "could not set lock on file" in err or "conflicting lock" in err:
        raise RuntimeError(
            "DuckDB file is locked by another process. Close other notebooks/python sessions using this DB, then rerun this cell."
        )
    raise RuntimeError("Pipeline run failed. Check output above.")

exit code: 0

STDOUT:
 Pipeline taxi_pipeline load step completed in 0.71 seconds
1 load package(s) were loaded to destination duckdb and into dataset taxi_data
The duckdb destination used duckdb:////Users/sahand/Desktop/DataTalks/DataEngineering2025/data-engineering-zoomcamp/cohorts/2026/workshops/dlt/taxi_pipeline_project/taxi_pipeline.duckdb location to store data
Load package 1774893910.086407 is LOADED and contains no failed jobs


STDERR:
 2026-03-30 12:05:31,073|[WARNING]|67035|8433342656|dlt|validate.py|verify_normalized_table:113|In schema `taxi`: The following columns in table 'ny_taxi' did not receive any data during this load and therefore could not have their types inferred:
  - mta_tax
  - rate_code

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'mta_tax': {'data_type': 'text'}})




## Step 3: Query results and answer homework questions

We connect directly to the pipeline DuckDB and compute:
1. dataset date range
2. % of trips paid by credit card
3. total tips

In [5]:
import duckdb
import pandas as pd

con = duckdb.connect(str(DUCKDB_FILE))

# Find user tables in any non-system schema (robust if schema changes)
tables = con.sql("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
ORDER BY table_schema, table_name
""").df()

print("Tables discovered:")
print(tables)

candidate = tables[
    (~tables["table_name"].str.startswith("_dlt"))
    & (~tables["table_name"].str.startswith("sqlite_"))
]

if candidate.empty:
    raise RuntimeError("No data tables found. Run pipeline cell first (Cell 7).")

schema_name = candidate.iloc[0]["table_schema"]
table_name = candidate.iloc[0]["table_name"]
table_ref = f'"{schema_name}"."{table_name}"'

print("\nUsing table:", table_ref)

Tables discovered:
  table_schema           table_name
0    taxi_data           _dlt_loads
1    taxi_data  _dlt_pipeline_state
2    taxi_data         _dlt_version
3    taxi_data              ny_taxi

Using table: "taxi_data"."ny_taxi"


In [6]:
# Q1–Q3: dlt normalizes API column names to snake_case in DuckDB
# (Trip_Pickup_DateTime -> trip_pickup_date_time, etc.)
q1 = con.sql(f"""
SELECT
  MIN(CAST(trip_pickup_date_time AS DATE)) AS start_date,
  MAX(CAST(trip_pickup_date_time AS DATE)) AS end_date
FROM {table_ref}
""").df()
print("Q1 result:")
print(q1)

# Q2: proportion paid with credit card (API uses "Credit")
q2 = con.sql(f"""
SELECT
  ROUND(100.0 * AVG(CASE WHEN lower(trim(payment_type)) = 'credit' THEN 1 ELSE 0 END), 2) AS credit_card_pct
FROM {table_ref}
""").df()
print("\nQ2 result:")
print(q2)

# Q3: total tips
q3 = con.sql(f"""
SELECT ROUND(SUM(tip_amt), 2) AS total_tips
FROM {table_ref}
""").df()
print("\nQ3 result:")
print(q3)

Q1 result:
  start_date   end_date
0 2009-06-01 2009-06-30

Q2 result:
   credit_card_pct
0            26.66

Q3 result:
   total_tips
0     6063.41


In [7]:
# Map computed values to homework options
start_date = pd.to_datetime(q1.loc[0, "start_date"]).date()
end_date = pd.to_datetime(q1.loc[0, "end_date"]).date()
end_exclusive = end_date + pd.Timedelta(days=1)
credit_card_pct = float(q2.loc[0, "credit_card_pct"])
total_tips = float(q3.loc[0, "total_tips"])

print("Homework answers:")
print(f"1) Date range (inclusive): {start_date} to {end_date}")
print(f"   Option format (end exclusive): {start_date} to {end_exclusive} -> 2009-06-01 to 2009-07-01")
print(f"2) Credit card proportion: {credit_card_pct}% -> option: 26.66%")
print(f"3) Total tips: ${total_tips:,.2f} -> option: $6,063.41")

Homework answers:
1) Date range (inclusive): 2009-06-01 to 2009-06-30
   Option format (end exclusive): 2009-06-01 to 2009-07-01 -> 2009-06-01 to 2009-07-01
2) Credit card proportion: 26.66% -> option: 26.66%
3) Total tips: $6,063.41 -> option: $6,063.41


## Step 4 (optional): inspect with dlt dashboard

From terminal in project folder:

```bash
cd cohorts/2026/workshops/dlt/taxi_pipeline_project
dlt pipeline taxi_pipeline show
```

This helps you verify tables, schema, and load metadata.

## What to learn from this homework

- How to implement pagination manually for APIs that are not scaffolded.
- How dlt runs extract -> normalize -> load in one `pipeline.run()`.
- How to validate homework answers with SQL directly from DuckDB.

If you want, next I can add challenge exercises (incremental loading + data quality checks) to this notebook.

## Final check (one-shot PASS/FAIL)

Run this after Cells 9-11. It validates the computed results against expected homework options.

In [8]:
# Final verification before submission
expected_start = pd.to_datetime("2009-06-01").date()
expected_end_exclusive = pd.to_datetime("2009-07-01").date()
expected_credit_pct = 26.66
expected_tips = 6063.41

actual_start = pd.to_datetime(q1.loc[0, "start_date"]).date()
actual_end_inclusive = pd.to_datetime(q1.loc[0, "end_date"]).date()
actual_end_exclusive = (pd.Timestamp(actual_end_inclusive) + pd.Timedelta(days=1)).date()
actual_credit_pct = round(float(q2.loc[0, "credit_card_pct"]), 2)
actual_tips = round(float(q3.loc[0, "total_tips"]), 2)

checks = {
    "Q1 date range option": (actual_start == expected_start and actual_end_exclusive == expected_end_exclusive),
    "Q2 credit card % option": (actual_credit_pct == expected_credit_pct),
    "Q3 total tips option": (actual_tips == expected_tips),
}

print("Submission check:")
for name, ok in checks.items():
    print(f"- {name}: {'PASS' if ok else 'FAIL'}")

if all(checks.values()):
    print("\nALL PASS - ready to submit.")
else:
    print("\nSome checks failed. Re-run cells 7 -> 11 and verify pipeline/data.")
    print("Actuals:")
    print(f"  start={actual_start}, end_exclusive={actual_end_exclusive}")
    print(f"  credit_pct={actual_credit_pct}, tips={actual_tips}")

Submission check:
- Q1 date range option: PASS
- Q2 credit card % option: PASS
- Q3 total tips option: PASS

ALL PASS - ready to submit.
